# MRAi_P3 — Fase 2 en Colab (GPU NVIDIA)
Fine-tune 2D-US elongado. Guía completa en `COLAB.md`. Ejecuta celdas en orden 0→6.

## 0 — GPU check
Si `GPUs: []`, cambia el runtime a GPU antes de seguir (Entorno → Tipo de ejecución).

In [ ]:
!ls /content/drive/MyDrive/mrai_p3/colab 2>/dev/null || echo 'monta Drive en celda 1 primero'
import sys; sys.path.insert(0, '/content/drive/MyDrive/mrai_p3/colab/tools')
exec(open('/content/drive/MyDrive/mrai_p3/colab/tools/verify_gpu.py').read())

## 1 — Setup: pip + Drive + rutas
Instala deps, monta Drive y define BASE/DATA/OUT en Drive (sobrevive al reciclaje de la VM).

In [ ]:
%pip install -q -r /content/drive/MyDrive/mrai_p3/colab/requirements-colab.txt
from google.colab import drive; drive.mount('/content/drive')
import os
BASE='/content/drive/MyDrive/mrai_p3'; COLAB=f'{BASE}/colab'; DATA='/content/dataset'; STAGED='/content/staged'; OUT=f'{BASE}/out_elong'
os.makedirs(OUT, exist_ok=True)
print(BASE, COLAB, DATA, STAGED, OUT)

## 2 — Stage data
Descomprime `dataset.zip` (spectra+labels+dataset.json+manifest.jsonl, ~2.5 GB) y `DNN_0.zip` (--init).

In [ ]:
!unzip -q -o /content/drive/MyDrive/mrai_p3/dataset.zip -d /content/dataset_tmp && mkdir -p /content/dataset && cp -r /content/dataset_tmp/*/* /content/dataset/ 2>/dev/null || cp -r /content/dataset_tmp/* /content/dataset/
!unzip -q -o /content/drive/MyDrive/mrai_p3/DNN_0.zip -d /content/init
!ls /content/dataset/ && ls /content/dataset/spectra | head -3 && ls /content/init/

## 3 — Ingesta Fase 1 (ruta jres)
Regenera `manifest.json` con `source=/content/dataset`. Primero `--validate-only` + probe, luego ingesta real.

In [ ]:
%cd /content/drive/MyDrive/mrai_p3/colab
!python scripts/jres_ingest.py --dataset /content/dataset --out /content/staged --validate-only
!python scripts/jres_ingest.py --dataset /content/dataset --out /content/staged
!cat /content/staged/manifest.json | head -c 600

## 4 — Smoke test
Valida modelo + I/O con 1 época pequeña antes de gastar horas.

In [ ]:
!python scripts/train_model.py --data /content/staged --max-pts 4096 --epochs 1 --batch 4096 --out /content/smoke

## 4b — Pre-extraccion: pool de fondo (acelerador, ~15 s una vez)
Guarda por espectro TODOS los positivos + `pool-k` x positivos de fondo. El submuestreo por epoca (`--bg-ratio`) lo hace el entrenamiento desde ese pool: el `split_2d` (~1 s/espectro) se paga UNA vez y el fondo SIGUE variando entre epocas.
Tamano: pool-k 10 -> ~3.3 GB/1000 espectros (0.9 GB con pool-k 2, 6.3 GB con pool-k 20). Debe ser mayor que el `--bg-ratio` de train para que haya variacion.
Con la config de Colab (`--batch 65536`, `--bg-ratio 4`) el sample por epoca son ~2905 puntos/espectro = 581 positivos + 2324 de fondo -> **35 steps train / 8 val** por epoca. Ojo: `M` depende de `bg_ratio`, NO de `batch` (`max-pts` nunca se alcanza). El pool (pool-k 10 = ~6391 filas) es mayor que ese sample, asi que el fondo varia entre epocas.
Medido en el Mac: 1000 espectros en ~11-16 s (antes 3.6 min) -> 3.0 GB en disco.
Solo se calculan las filas del pool (`split_2d_rows`): la matriz completa de 524288 filas no se construye y la RAM pico por espectro baja de 1.39 GB a 0.17 GB.

In [ ]:
!python scripts/preextract.py --dataset /content/dataset --ids /content/staged/manifest.json --out /content/pre --seed 0 --pool-k 10 --fast
!echo '--- log (ultimas) ---' && tail -n 3 /content/pre/preextract.log
!du -sh /content/pre && ls /content/pre/*.npz | wc -l

## 5 — Fine-tune (recomendado)
Usa `/content/pre` pre-extraido (celda 4b): ~5-10 s/epoca con 35 steps train (`--batch 65536`, `--bg-ratio 4`) + early-stopping; 30-50 epocas ≈ 4-8 min.
Ruta on-demand (sin pre-extraer, replica literal): cambia `--data /content/pre --preextracted` por `--data /content/staged`. Paga `split_2d` en CADA epoca: en el Mac son ~18 s/epoca con el split por filas (default; `--slow-split` fuerza la matriz completa, ~3.5 min/epoca), y en la CPU del runtime de Colab sera mas lento.
El prep de datos on-demand son ~3.4 s/epoca frente a ~0.8 s con el pool (~20% vs ~5% de una epoca de ~15 s), asi que pre-extraer sigue pagandose solo a partir de ~4 epocas, pero ya no es obligatorio.
Salida en Drive + `ckpt.keras` para reanudar si Colab corta.
`--scratch /content/out` escribe en local y copia a Drive cada 5 epocas: escribir en Drive cada epoca cuesta 0.3-20 s y hacia que las epocas oscilasen entre 1 s y 20 s con el step constante.

In [ ]:
!python scripts/train_model.py --data /content/pre --preextracted --init /content/init/DNN_0 --lr 1e-4 --batch 65536 --bg-ratio 4 --epochs 210 --patience 20 --out /content/drive/MyDrive/mrai_p3/out_elong --scratch /content/out --sync-every 5

## 5b — Reanudar tras corte (solo si Colab se desconecto)
`ckpt.keras` + `training.log` ya estan en Drive (la celda 5 escribe ahi). Esta celda continua desde el mejor checkpoint.
**OJO 1 — `/content` se vacia al desconectarse Colab** (solo sobrevive Drive). Re-ejecuta ANTES, en orden: celda 1 (setup) -> 2 (stage data) -> 3 (ingesta) -> 4b (pre-extraccion). Tarda ~5 min y es obligatorio: sin `/content/pre` esta celda falla.
**OJO 2 — `CSVLogger` sobrescribe `training.log`**: la celda lo renombra antes a `training_part1.log` para no perder la curva.
**OJO 3 — no re-ejecutes la celda 5 tras un corte**: usa `--init` y empezaria de cero.

In [ ]:
!ls -lh /content/drive/MyDrive/mrai_p3/out_elong/ckpt.keras && tail -n 5 /content/drive/MyDrive/mrai_p3/out_elong/training.log
!mv /content/drive/MyDrive/mrai_p3/out_elong/training.log /content/drive/MyDrive/mrai_p3/out_elong/training_part1.log 2>/dev/null; echo log-rotated
!python scripts/train_model.py --data /content/pre --preextracted --resume /content/drive/MyDrive/mrai_p3/out_elong/ckpt.keras --lr 1e-4 --batch 65536 --bg-ratio 4 --epochs 210 --patience 20 --out /content/drive/MyDrive/mrai_p3/out_elong --scratch /content/out --sync-every 5

## 6 — Export
Empaqueta el resultado para bajar al Mac y usar con `StoP3` sin cambios.

In [ ]:
!cd /content/drive/MyDrive/mrai_p3 && rm -f out_elong.zip && zip -qr out_elong.zip out_elong && ls -lh out_elong.zip
!ls /content/drive/MyDrive/mrai_p3/out_elong/